# Hopfield 统一任务基准

状态：`shared-components / numerical-validation-pending`

在 Colab 打开这一个 Notebook 即可。环境单元会自动下载固定提交的共享代码，不需要手动上传多个文件，也不额外安装依赖。六个模型及 a/c/e/f 已移到 `am_bench/`，下面保留问题、配置与图表入口；论文专属补充实验仍在本文件。

仓库未保存本次执行输出。公共自检包含拆分前后 36 个小样本的数值对照；通过才继续主实验。

**首次建议打开 [Q04：反复检索](https://colab.research.google.com/github/Heptazero/nn-labs/blob/main/hopfield-benchmark/retrieval_dynamics.ipynb)**：默认三个模型、432 次检索，支持分批恢复。本综合入口有更多附加扫描，主实验结束后才保存，暂不支持整份 Notebook 断点恢复。

## 0. 组件式实验管线与任务边界

`a 记忆输入 → b 模型存储 → c 检索线索 → d 检索动力学 → e 测量 → f 同图比较`

共享任务复用 `a/c/e/f`，只替换 `b/d`：U1 固定点、U2 噪声恢复、U3 有限容量、U4 吸引域、U5 动力学、U6 虚假吸引子、U7 资源效率。所有模型在 U2/U3/U4 的共同主指标是 **Top-1 memory identification**（终态与哪一条已存记忆最相似），因为它同时适用于二值终态和连续终态。

论文专属机制不混入总排名：Simplicial 做 H1 阶数、H2 随机结构、H3 同参数预算与 H7 结构×相似度；Curved/Explosive 做 H4 状态反馈有效温度和 H5 正反向迟滞；PSHN 做 H6 分组数与 feature-to-prototype。1982 的非对称、遗忘、同步周期和 1985 的有限温度相图/AT 线仍属于各自论文 notebook，避免为同一任务重复造轮子。

In [ ]:
# [环境] Colab 预装依赖；自动获取固定版本的共享代码。
# 只需打开本 Notebook，不需要手动上传多个 Python 文件。
from pathlib import Path
import subprocess
import sys
import tempfile

CODE_REV = "b3d54f01a9cc890717abaab47af2b3f0768a82b6"
cache_root = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
REPO = cache_root / ("nn-labs-" + CODE_REV[:12])
if not REPO.exists():
    subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
    subprocess.run(["git", "remote", "add", "origin", "https://github.com/Heptazero/nn-labs.git"],
                   cwd=REPO, check=True, capture_output=True)
# 未完成的下载可重试；已有不同 checkout 则拒绝覆盖。
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
if head.returncode:
    subprocess.run(["git", "fetch", "--depth=1", "origin", CODE_REV], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != CODE_REV:
    raise RuntimeError("缓存 checkout 版本不符；请保留改动并使用新运行时。")
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("共享代码已有修改；请另存修改并使用干净运行时完成基线验收。")
if "am_bench" in sys.modules and Path(sys.modules["am_bench"].__file__).resolve().parent != REPO / "am_bench":
    raise RuntimeError("已载入其他版本的 am_bench；请重启会话后运行。")
sys.path.insert(0, str(REPO))
import am_bench
assert am_bench.API_VERSION == 1
from am_bench.provenance import source_info
SOURCE = source_info()
assert SOURCE["git_commit"] == CODE_REV
print("Loaded shared code:", SOURCE)

# [环境] Colab 预装依赖；自动获取固定版本的共享代码。
# 只需打开本 Notebook，不需要手动上传多个 Python 文件。
from pathlib import Path
import subprocess
import sys
import tempfile

CODE_REV = "b3d54f01a9cc890717abaab47af2b3f0768a82b6"
cache_root = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
REPO = cache_root / ("nn-labs-" + CODE_REV[:12])
if not REPO.exists():
    subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
    subprocess.run(["git", "remote", "add", "origin", "https://github.com/Heptazero/nn-labs.git"],
                   cwd=REPO, check=True, capture_output=True)
# 未完成的下载可重试；已有不同 checkout 则拒绝覆盖。
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
if head.returncode:
    subprocess.run(["git", "fetch", "--depth=1", "origin", CODE_REV], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual != CODE_REV:
    raise RuntimeError("缓存 checkout 版本不符；请保留改动并使用新运行时。")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip():
    raise RuntimeError("共享代码已有修改；请另存修改并使用干净运行时完成基线验收。")
if "am_bench" in sys.modules and Path(sys.modules["am_bench"].__file__).resolve().parent != REPO / "am_bench":
    raise RuntimeError("已载入其他版本的 am_bench；请重启会话后运行。")
sys.path.insert(0, str(REPO))
import am_bench
assert am_bench.API_VERSION == 1
from am_bench.provenance import source_info
SOURCE = source_info()
assert SOURCE["git_commit"] == CODE_REV
print("Loaded shared code:", SOURCE)

# [环境] 只使用 Colab 预装的 Python/PyTorch/绘图库，不在 notebook 里安装依赖
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from time import perf_counter
from typing import Any, Callable, Iterable
from urllib.request import Request, urlopen
from uuid import uuid4
import json

# [展示] Matplotlib/Pandas 只消费标准结果表，不参与模型更新
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

# [展示] 全局画图风格固定，避免不同实验各自偷偷改视觉编码
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)


SOURCE_NOTEBOOK = "hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb"
SOURCE_COMMIT = SOURCE["git_commit"]  # 实际加载的共享代码版本，不是远端最新 main。
print(f"torch={torch.__version__}, pandas={pd.__version__}")
SOURCE_NOTEBOOK = "hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb"
SOURCE_COMMIT = SOURCE["git_commit"]  # 实际加载的共享代码版本，不是远端最新 main。
print(f"torch={torch.__version__}, pandas={pd.__version__}")

## 1. a / c：共享记忆与共享线索

`a1` 生成独立随机 `{-1,+1}` 模式。`c1` 精确翻转 `round(rho*N)` 个位置；`c2` 生成与记忆独立的随机初态。线索始终在模型循环外生成，同一 trial 的所有模型收到逐元素相同的 tensor。

In [ ]:
# a/c：共享数据与查询；定义在仓库 am_bench/tasks.py。
from am_bench.tasks import MemorySet, make_generator, a1_make_independent_binary, c1_make_hamming_cue, c2_make_random_state

## 2. b / d：可替换模型组件

所有适配器实现 `fit → retrieve → resource_summary`，但保留各自原生状态与动力学。

- Classical Hopfield 存储 `W=(1/N)Σ ξξᵀ` 且清零对角线，随机顺序异步更新。
- Polynomial DAM 使用 `E=-Σμ mμ^d`；Exponential DAM 使用负 log-sum-exp 能量代理，二者逐坐标比较 `s_i=+1/-1` 的能量。
- Simplicial R12 使用 `E=-Σσ wσ Sσ`，其中 `wσ=(1/N)Σμ ξσμ`；主适配器以边和三元单形混合，并用异步坐标下降保证可检查的能量轨迹。论文原生同步规则只在专属解释中讨论。
- PSHN 将坐标分成 `k` 组，一次读出时把“其余组相关度的乘积”作为每组记忆系数。
- Continuous Modern Hopfield 使用 `softmax(beta·Xq)X` 返回连续状态，不伪装成二值固定点动力学。

这些能量数值不共享同一零点或尺度，因此 U5 只同图比较公共误差轨迹，不把不同能量直接画成高低排名。

In [ ]:
# b/d：每个模型族一个文件；target 只进入外部测量层。
from am_bench.models import (RetrievalResult, ClassicalHopfield, PolynomialDAM,
                             ExponentialDAM, SimplicialR12, PSHN,
                             ContinuousModernHopfield, MODEL_FACTORIES)
from am_bench.models.base import binary_state
MODEL_FACTORIES

## 3. e：统一测量与结果协议

测量函数只读取目标、终态、轨迹和资源记录，不根据模型名称偷偷换判据。共同图使用 Top-1；二值模型额外报告 exact recall。U1 的固定点判据是干净记忆经过一次登记更新后完全不变；不具备离散固定点语义的连续模型会显式标为不适用，而不是记作失败。

In [ ]:
# e：统一终态测量和独立轨迹观察器。
from am_bench.metrics import (e1_exact_recall, e2_overlap, e3_top1_memory,
                              e4_attractor_class, retrieve_measured)

## 4. 配对运行器

扫描范围、重复数和停止上限都在运行前固定。异常、达到最大步数和删失不会被静默删除。`run_id + case fields` 相同的记录必须包含全部模型。

In [ ]:
from am_bench.runner import (BenchmarkConfig, stable_seed, PAIR_KEY,
                             validate_paired_results, failure_row,
                             run_paired_benchmark, capacity_summary)

## 5. f：同图比较组件

颜色只编码模型。共同曲线默认读取 `top1_correct`，因此连续输出不会被强制二值化。容量图只纳入 fixed-point-eligible 记录并使用实际 `P`；失败保留在分母，扫描边界用删失符号表示。

In [ ]:
from am_bench.plots import (MODEL_LABELS, curve_table, plot_success_curve,
                            plot_capacity, plot_dynamics, plot_quality_cost)

## 6. 公共组件自检

这不是实验结论。它只验证六个适配器能接收同一个 memory/cue、返回统一字段；若模型提供离散能量轨迹，再检查该轨迹不增加。Continuous Modern 和 PSHN 不伪造能量曲线。

In [ ]:
# [验收] 对比冻结的拆分前版本；下载内容先核对 SHA-256，再执行指定模型定义。
NUMERICAL_GATE_PASSED = False
from urllib.request import urlopen
import hashlib
from am_bench.validation import verify_extraction, verify_dynamics

REFERENCE_REV = "e9adf216c5b5a0b329a102c0c25954e6ff48aa12"
reference = cache_root / ("nn-labs-reference-" + REFERENCE_REV[:12] + ".ipynb")
reference_url = ("https://raw.githubusercontent.com/Heptazero/nn-labs/" + REFERENCE_REV
                 + "/hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb")
if not reference.exists():
    with urlopen(reference_url, timeout=60) as response:
        reference.write_bytes(response.read())
if hashlib.sha256(reference.read_bytes()).hexdigest() != "7568d0bfe28703f24026dd49fc2756ecb7f58141ae2484e29652abace6d85be0":
    raise RuntimeError("旧版参考文件摘要不符；请删除该缓存文件后重新下载。")
print(verify_extraction(reference))
print(verify_dynamics())
NUMERICAL_GATE_PASSED = True

# [验收] 对比冻结的拆分前版本；下载内容先核对 SHA-256，再执行指定模型定义。
NUMERICAL_GATE_PASSED = False
from urllib.request import urlopen
import hashlib
from am_bench.validation import verify_extraction, verify_dynamics

REFERENCE_REV = "e9adf216c5b5a0b329a102c0c25954e6ff48aa12"
reference = cache_root / ("nn-labs-reference-" + REFERENCE_REV[:12] + ".ipynb")
reference_url = ("https://raw.githubusercontent.com/Heptazero/nn-labs/" + REFERENCE_REV
                 + "/hopfield-benchmark/hopfield_benchmark_phase1_colab.ipynb")
if not reference.exists():
    with urlopen(reference_url, timeout=60) as response:
        reference.write_bytes(response.read())
if hashlib.sha256(reference.read_bytes()).hexdigest() != "7568d0bfe28703f24026dd49fc2756ecb7f58141ae2484e29652abace6d85be0":
    raise RuntimeError("旧版参考文件摘要不符；请删除该缓存文件后重新下载。")
print(verify_extraction(reference))
print(verify_dynamics())
NUMERICAL_GATE_PASSED = True

# [自检输入] 小尺寸 memory/cue 只检查组件契约，不进入任何实验结论
check_memories = a1_make_independent_binary(N=32, P=4, data_seed=7)
check_target = check_memories.patterns[0]
check_cue = c1_make_hamming_cue(check_target, 0.125, cue_seed=11)
for registered_id, factory in MODEL_FACTORIES.items():
    check_model = factory().fit(check_memories.patterns)
    assert check_model.model_id == registered_id
    check_result = retrieve_measured(
        check_model, check_cue, target=check_target, update_seed=13, max_sweeps=8
    )
    # [判断] 只有模型真实提供两点以上能量轨迹时才检查单调性
    if len(check_result.energy_trace) > 1:
        deltas = torch.diff(torch.tensor(check_result.energy_trace))
        assert bool(torch.all(deltas <= 1e-10))
        energy_check = "monotone"
    else:
        energy_check = "not supplied"
    print(registered_id, check_result.status, check_result.sweeps, energy_check)
print("component contract: passed")

## 7. 有限扫描配置

扫描范围、重复数与停止上限预先固定。它足以验证统一接口和有限规模趋势，不足以估计渐近容量阶数。主图是 `native / equal_max_sweeps`；U7 显式展示资源差异，H1–H3 另做同参数的高阶结构消融。

In [ ]:
# [溯源] 每次 Colab Run All 生成唯一 execution_id，避免覆盖后无法区分批次
EXECUTION_ID = (
    datetime.now(timezone.utc).strftime("benchmark-v1-%Y%m%dT%H%M%SZ-")
    + uuid4().hex[:8]
)
# [实验控制] 有限扫描 Gate；这里改参数会直接改变结论适用范围
CONFIG = BenchmarkConfig(
    N_values=(64, 128),
    P_values=(4, 8, 16, 32),
    corruption_levels=(0.0, 0.1, 0.2, 0.3),
    pattern_sets=3,
    targets_per_set=4,
    max_sweeps=20,
    base_seed=20260905,
    experiment_id=EXECUTION_ID,
    source_commit=SOURCE_COMMIT,
)
display(pd.Series(asdict(CONFIG), name="value").to_frame())

## 8. 运行并保存原始记录

每一行是一条目标记忆的一次检索。原始输出保存为 JSON Lines，列表轨迹不会被 CSV 静默改形。失败记录仍保留在比较分母中。

In [ ]:
assert NUMERICAL_GATE_PASSED
assert NUMERICAL_GATE_PASSED
# [执行] 这是主数值入口；本仓库提交的 notebook 保持未执行、无缓存输出
results = run_paired_benchmark(MODEL_FACTORIES, CONFIG)
validate_paired_results(results, tuple(MODEL_FACTORIES))

artifact_root = cache_root / "hopfield-benchmark-results" / EXECUTION_ID
artifact_root.mkdir(parents=True, exist_ok=True)
# [存储] JSON Lines 保留每条 trial 和列表轨迹，避免 CSV 把列表静默字符串化
results.to_json(
    artifact_root / "benchmark_v1_raw_results.jsonl",
    orient="records", lines=True,
)
with (artifact_root / "benchmark_v1_config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(CONFIG), handle, ensure_ascii=False, indent=2)
print(f"saved {len(results)} rows to {artifact_root}")
display(results.drop(columns=["energy_trace", "error_trace"]).head(8))
display(results.groupby(["model_id", "status"]).size().rename("count").to_frame())
(artifact_root / "source.json").write_text(json.dumps(SOURCE, indent=2))
(artifact_root / "source.json").write_text(json.dumps(SOURCE, indent=2))

## 实验 1：U2 噪声恢复

固定 `N=128, P=16`。横轴是逐元素相同的 Hamming 损坏线索，纵轴是适用于二值与连续输出的 Top-1 记忆识别率。

In [ ]:
# [汇总] 固定 N/P 后只让 corruption_level 变化，形成 U2 配对噪声曲线
noise_curve = curve_table(results, "corruption_level", {"N": 128, "P": 16})
plot_success_curve(
    noise_curve, "corruption_level", "Hamming corruption ratio",
    "U2 noise recovery | N=128, P=16 | native budget",
)
plt.show()

In [ ]:
endpoint = noise_curve[noise_curve["corruption_level"] == 0.3]
# [观测·数据] Top-1 回答认出哪条记忆，overlap 补充终态离目标还有多远
endpoint_overlap = (
    results[
        (results["N"] == 128)
        & (results["P"] == 16)
        & (results["corruption_level"] == 0.3)
    ]
    .groupby("model_id")["overlap"].mean()
)
lines = [
    f"- `{row['model_id']}`：rho=0.30 时 Top-1={row['mean']:.3f}，"
    f"final overlap={endpoint_overlap.loc[row['model_id']]:.3f}，"
    f"pattern-set replicates={int(row['count'])}。"
    for _, row in endpoint.iterrows()
]
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这里只是有限规模、native budget 的配对结果；连续输出没有被强制二值化，不能据此宣称容量阶数或统计优势。"
))

## 实验 2：U3 有限负载曲线

固定 `N=128` 和 10% Hamming 损坏。横轴使用实际存储数 `P`，不把线性、多项式和指数容量强行归一成同一个 `P/N`。

In [ ]:
# [汇总] 固定 N/rho 后扫描实际 P；不同容量阶的模型不强行共用 P/N
load_curve = curve_table(results, "P", {"N": 128, "corruption_level": 0.1})
plot_success_curve(
    load_curve, "P", "Stored patterns P",
    "U3 finite-load curve | N=128, corruption=0.10 | native budget",
    log_x=True,
)
plt.show()

In [ ]:
display(Markdown(
    "**本次运行的描述性结论**：下表是上图对应的条件均值。\n\n"
    "**证据边界**：曲线可能非单调；不在预设扫描点之间插值，也不把最后一个成功点自动称为理论容量。"
))
display(load_curve.pivot(index="model_id", columns="P", values="mean").round(3))

## 实验 3：U1/U3 干净固定点容量

采用预先固定的 90% one-update-unchanged 判据，只纳入具备该语义的模型。向上空心点表示容量至少达到扫描上界；向下空心点表示容量低于或等于扫描下界。

In [ ]:
# [汇总] 阈值在看结果前固定为 0.9，避免事后移动门槛
capacities = capacity_summary(results, threshold=0.9)
capacities.to_csv(artifact_root / "benchmark_v1_capacity_summary.csv", index=False)
display(capacities)
plot_capacity(capacities)
plt.show()

In [ ]:
lines = []
for row in capacities.itertuples(index=False):
    relation = "≤" if row.left_censored else "≥" if row.right_censored else "="
    lines.append(f"- `{row.model_id}`，N={row.N}：P_c {relation} {row.P_c}。")
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：这是离散扫描上的经验阈值。删失点不能用于普通容量拟合，也不能支持线性、多项式或指数容量结论。"
))

## 实验 4：U5 配对动力学

同一个 `run_id` 下叠加公共 bit-error 指标。两种模型的能量定义不同，所以不把能量数值混在一个纵轴。

In [ ]:
# [实验控制] 先选定一条完整配对 trial，再叠加六个模型的轨迹
dynamics_run_id = results[
    (results["N"] == 128)
    & (results["P"] == 16)
    & (results["corruption_level"] == 0.2)
]["run_id"].iloc[0]
plot_dynamics(results, dynamics_run_id)
plt.show()

In [ ]:
lines = []
for row in results[results["run_id"] == dynamics_run_id].itertuples(index=False):
    final_error = row.error_trace[-1] if row.error_trace else float("nan")
    lines.append(
        f"- `{row.model_id}`：status={row.status}，sweeps={row.sweeps}，final bit error={final_error:.3f}。"
    )
display(Markdown(
    "**本次运行的描述性结论**\n\n" + "\n".join(lines)
    + "\n\n**证据边界**：单条轨迹只解释更新过程，不代表总体召回率。"
))

## 实验 5：U7 质量—计算量

横轴是实现登记的近似操作数，不是硬件实测延迟。此图揭示 native 配置的资源差异；H1–H3 才是明确的同参数结构比较。

In [ ]:
plot_quality_cost(results, N=128, P=16, level=0.1)
plt.show()

In [ ]:
# [观测·资源] 同时列参数数、真实存储字节和检索 FLOPs，wall time 不作硬件结论
resource_table = (
    results[
        (results["N"] == 128)
        & (results["P"] == 16)
        & (results["corruption_level"] == 0.1)
    ]
    .groupby("model_id", as_index=False)
    .agg(
        top1_correct=("top1_correct", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
        storage_bytes=("storage_bytes", "mean"),
        parameter_count=("parameter_count", "mean"),
    )
)
display(Markdown(
    "**本次运行的描述性结论**：同图中的质量差异必须和下面的参数、存储与计算量一起读。\n\n"
    "**证据边界**：当前不做硬件速度结论；向量化程度会改变 wall-clock time。"
))
display(resource_table.round(3))

## 实验 6：U4 吸引域

U4 复用 U2 的完整噪声曲线，但汇总量不同：AUC 是扫描区间内曲线下面积，`rho@90%` 是仍保持至少 90% Top-1 成功率的最大已扫描噪声。没有跨扫描点外推。

In [ ]:
# [汇总] U4 不重复运行模型，直接从完整 U2 曲线计算 AUC 与 rho@90%
def basin_summary(curve: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    rows = []
    for model_id, group in curve.groupby("model_id", sort=False):
        ordered = group.sort_values("corruption_level")
        x = ordered["corruption_level"].to_numpy(dtype=float)
        y = ordered["mean"].to_numpy(dtype=float)
        # [判断] 只取已扫描且成功率≥阈值的噪声点，不在网格之间插值
        passing = x[y >= threshold]
        rows.append({
            "model_id": model_id,
            # [观测·数据] AUC 只覆盖当前 corruption 网格的横轴区间
            "auc": float(np.trapz(y, x)),
            "rho_at_90": float(passing.max()) if passing.size else float("nan"),
            "right_censored": bool(passing.size and passing.max() == x.max()),
        })
    return pd.DataFrame(rows)


basins = basin_summary(noise_curve)
display(basins.round(3))
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(
        f"- `{row.model_id}`：AUC={row.auc:.3f}，rho@90%={row.rho_at_90:.2f}"
        + ("（达到扫描上界）" if row.right_censored else "") + "。"
        for row in basins.itertuples(index=False)
    )
    + "\n\n**证据边界**：AUC 只覆盖当前 rho 网格；rho@90% 是经验扫描值，不是理论吸引域半径。"
))

## 实验 7：U6 随机初态与虚假吸引子

虚假吸引子需要“真的迭代到固定点”才有含义，因此这里只比较 Classical、Polynomial DAM、Exponential DAM 与异步 Simplicial R12。PSHN 的一次映射和 Continuous Modern 的连续读出不被伪装成吸引子动力学。

In [ ]:
# [适用边界] U6 只纳入真正迭代到 fixed 的离散模型
ATTRACTOR_FACTORIES = {
    key: MODEL_FACTORIES[key]
    for key in [
        "classical_hebb", "polynomial_dam_d3",
        "exponential_dam", "simplicial_r12_t50",
    ]
}


# [观测·分类] 随机初态终点分为 stored、inverse、spurious 与 nonconverged
def classify_random_attractor(
    state: torch.Tensor, memories: torch.Tensor, status: str
) -> str:
    # [判断] 没到固定点时不根据暂态外观猜测吸引子类别
    if status != "fixed":
        return "nonconverged"
    final = state.to(torch.int8)
    stored = memories.to(torch.int8)
    if bool(torch.any(torch.all(stored == final.unsqueeze(0), dim=1))):
        return "stored_memory"
    if bool(torch.any(torch.all(stored == -final.unsqueeze(0), dim=1))):
        return "inverse_memory"
    return "spurious_fixed"


# [执行] 每个随机初态在模型循环外生成，再交给全部离散适配器
def run_random_start_benchmark(
    factories: dict[str, Callable[[], Any]],
    *, N: int, P: int, pattern_sets: int, starts_per_set: int,
    max_sweeps: int, base_seed: int,
) -> pd.DataFrame:
    rows = []
    for set_index in range(pattern_sets):
        data_seed = stable_seed(base_seed, set_index)
        memories = a1_make_independent_binary(N, P, data_seed)
        # [存储] 同一 pattern set 下复用已拟合模型，随机 start 只改变 cue
        fitted = {key: factory().fit(memories.patterns) for key, factory in factories.items()}
        for start_index in range(starts_per_set):
            cue_seed = stable_seed(base_seed, set_index, start_index, 17)
            update_seed = stable_seed(cue_seed, 91)
            # [输入] cue 与存储记忆独立；cue_seed 对所有模型完全相同
            cue = c2_make_random_state(N, cue_seed)
            for model_id, model in fitted.items():
                result = retrieve_measured(
                    model,
                    cue, target=memories.patterns[0],
                    update_seed=update_seed, max_sweeps=max_sweeps,
                )
                rows.append({
                    "model_id": model_id,
                    "pattern_set_id": set_index,
                    "start_id": start_index,
                    "data_seed": data_seed,
                    "cue_seed": cue_seed,
                    "update_seed": update_seed,
                    "outcome": classify_random_attractor(
                        result.final_state, memories.patterns, result.status
                    ),
                })
    return pd.DataFrame(rows)


spurious_results = run_random_start_benchmark(
    ATTRACTOR_FACTORIES,
    N=128, P=16, pattern_sets=3, starts_per_set=24,
    max_sweeps=30, base_seed=20260906,
)
spurious_results.to_json(
    artifact_root / "u6_random_start_raw.jsonl", orient="records", lines=True
)
# [汇总] 失败/未收敛仍在总数中，四类比例按 model_id 各自归一到 1
spurious_rates = (
    spurious_results.groupby(["model_id", "outcome"], as_index=False)
    .size().rename(columns={"size": "count"})
)
spurious_rates["rate"] = (
    spurious_rates["count"]
    / spurious_rates.groupby("model_id")["count"].transform("sum")
)
pivot = spurious_rates.pivot(index="model_id", columns="outcome", values="rate").fillna(0)
order = ["stored_memory", "inverse_memory", "spurious_fixed", "nonconverged"]
pivot = pivot.reindex(columns=order, fill_value=0)
pivot.plot(kind="bar", stacked=True, figsize=(8.2, 4.8), ylim=(0, 1))
plt.ylabel("Outcome proportion")
plt.title("U6 random-start attractor outcomes | N=128, P=16")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
display(Markdown(
    "**本次运行的描述性结论**：随机初态的完整分类见下表；stored、inverse、spurious 与未收敛之和为 1。\n\n"
    "**证据边界**：这测的是当前有限 N、P 和停止上限下的经验吸引子分布，不等于热力学自旋玻璃相比例。"
))
display(pivot.round(3))

## 实验 8：H1/H2/H3 Simplicial 结构消融

仅改变边与三元单形的配比，并用多个 `structure_seed` 暴露随机拓扑方差。三个分面分别匹配 sparse pairwise R12 基线的连接权重数、显式索引+权重字节、以及每轮连接 incidence；compute 分面固定只做一轮更新，因此实际登记 FLOPs 也受同一上限约束。这里采用逐点异步下降扩展，**不是论文使用的同步更新复刻**。

In [ ]:
# [执行] H1/H2/H3 同时交叉阶数配比、拓扑种子和资源预算
def run_simplicial_ablation(
    *, N: int, P: int, corruption_level: float,
    fractions: tuple[float, ...], structure_seeds: tuple[int, ...],
    budget_types: tuple[str, ...], pattern_sets: int,
    targets_per_set: int, max_sweeps: int, base_seed: int,
) -> pd.DataFrame:
    rows = []
    for set_index in range(pattern_sets):
        data_seed = stable_seed(base_seed, set_index)
        # [输入] 一个 set 内所有 fraction/seed/budget 共享同一批模式
        memories = a1_make_independent_binary(N, P, data_seed)
        for structure_seed in structure_seeds:
            # [存储] structure_seed 只改变连接位置，不改变模式或 cue
            models = {
                (budget_type, fraction): SimplicialR12(
                    fraction, structure_seed, budget_type
                ).fit(memories.patterns)
                for budget_type in budget_types
                for fraction in fractions
            }
            for target_id in range(min(P, targets_per_set)):
                target = memories.patterns[target_id]
                cue_seed = stable_seed(base_seed, set_index, target_id, 37)
                # [输入] cue 位于所有结构条件循环外，保证结构消融严格配对
                cue = c1_make_hamming_cue(target, corruption_level, cue_seed)
                update_seed = stable_seed(cue_seed, 91)
                for (budget_type, fraction), model in models.items():
                    # [实验控制] matched-compute 固定一轮；另外两种预算允许按固定点停止
                    sweep_limit = 1 if budget_type == "compute" else max_sweeps
                    result = retrieve_measured(
                    model,
                        cue, target=target, update_seed=update_seed,
                        max_sweeps=sweep_limit,
                    )
                    resources = model.resource_summary()
                    # [观测·资源] 2E+3T 是完整一轮访问的单形顶点次数
                    incidences = 2 * len(model.edges) + 3 * len(model.triangles)
                    rows.append({
                        "budget_type": budget_type,
                        "triangle_fraction": fraction,
                        "structure_seed": structure_seed,
                        "pattern_set_id": set_index,
                        "target_id": target_id,
                        "top1_correct": e3_top1_memory(
                            result.final_state, memories.patterns
                        ) == target_id,
                        "parameter_count": resources["parameter_count"],
                        "storage_bytes": resources["storage_bytes"],
                        "incidences_per_sweep": incidences,
                        "retrieval_flops": result.retrieval_flops,
                    })
    return pd.DataFrame(rows)


simplicial_ablation = run_simplicial_ablation(
    N=64, P=16, corruption_level=0.2,
    fractions=(0.0, 0.25, 0.5, 0.75, 1.0),
    structure_seeds=(101, 202, 303, 404, 505),
    budget_types=("parameter", "storage", "compute"),
    pattern_sets=3, targets_per_set=4, max_sweeps=20,
    base_seed=20260907,
)
simplicial_ablation.to_json(
    artifact_root / "h123_simplicial_raw.jsonl", orient="records", lines=True
)
# [汇总] 先在每个结构种子内平均，再用种子间标准差显示拓扑敏感性
seed_rates = (
    simplicial_ablation.groupby(
        ["budget_type", "triangle_fraction", "structure_seed"], as_index=False
    )["top1_correct"].mean()
)
summary = (
    seed_rates.groupby(["budget_type", "triangle_fraction"])["top1_correct"]
    .agg(["mean", "std"]).reset_index()
)
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2), sharey=True)
for axis, budget_type in zip(axes, ("parameter", "storage", "compute")):
    panel = summary[summary["budget_type"] == budget_type]
    axis.errorbar(
        panel["triangle_fraction"], panel["mean"], yerr=panel["std"],
        marker="o", capsize=4,
    )
    axis.set(
        xlabel="Triangle fraction", title=f"matched {budget_type}",
        ylim=(-0.03, 1.03),
    )
axes[0].set_ylabel("Top-1 identification rate")
fig.suptitle("H1/H2/H3 Simplicial ablation")
fig.tight_layout()
plt.show()

# [判断] 图后公开三个预算的实际资源数，并用断言阻止整数取整越界
budgets = simplicial_ablation.groupby(
    ["budget_type", "triangle_fraction"], as_index=False
).agg(
    parameter_count=("parameter_count", "first"),
    storage_bytes=("storage_bytes", "first"),
    incidences_per_sweep=("incidences_per_sweep", "first"),
    retrieval_flops=("retrieval_flops", "first"),
)
parameter_panel = budgets[budgets["budget_type"] == "parameter"]
storage_panel = budgets[budgets["budget_type"] == "storage"]
compute_panel = budgets[budgets["budget_type"] == "compute"]
# [约束] 三条断言分别验证 matched-parameter/storage/compute 的承诺
assert parameter_panel["parameter_count"].nunique() == 1
assert storage_panel["storage_bytes"].max() <= storage_panel.iloc[0]["storage_bytes"]
assert compute_panel["incidences_per_sweep"].max() <= compute_panel.iloc[0]["incidences_per_sweep"]
assert compute_panel["retrieval_flops"].max() <= compute_panel.iloc[0]["retrieval_flops"]
display(Markdown(
    "**本次运行的描述性结论**：每个分面的均值比较阶数配比，误差条是随机拓扑种子的标准差；预算断言已通过。\n\n"
    "**证据边界**：匹配方式改变可用连接数；三个分面回答不同反事实，不能把最佳点跨分面拼成一个模型。"
))
display(summary.round(3))
display(budgets)

## 实验 9：H7 结构 × 相似度的完整 2×2

这是机制实验，不进入主模型排名。因素 A 是 pairwise 与 mixed simplicial 结构；因素 B 是 dot 与 cosine 相似度。四格共享相同模式、线索、单形数量和温度；在概率尺度上报告交互项 `p11 - p10 - p01 + p00`。这是受 Simplicial Hopfield 论文启发的协议扩展，不冒充论文原图。

In [ ]:
# [机制模型] H7 专用连续读出，只服务 2×2 结构×相似度反事实
class ContinuousSimplicialAttention:
    def __init__(
        self, triangle_fraction: float, similarity: str,
        structure_seed: int, beta: float = 1.0,
    ) -> None:
        self.triangle_fraction = float(triangle_fraction)
        self.similarity = similarity
        self.structure_seed = int(structure_seed)
        self.beta = float(beta)

    def fit(self, patterns: torch.Tensor) -> "ContinuousSimplicialAttention":
        self.patterns = patterns.to(torch.float64)
        self.P, self.N = map(int, self.patterns.shape)
        # [实验控制] pairwise 与 mixed 都使用同样数量的单形项
        budget = self.N * (self.N - 1) // 2
        triangle_count = int(round(self.triangle_fraction * budget))
        edge_count = budget - triangle_count
        generator = make_generator(self.structure_seed)
        # [中介变量] mixed 条件随机抽边/三角形；pairwise 条件使用完整边集
        all_edges = torch.combinations(torch.arange(self.N), r=2)
        self.edges = all_edges[torch.randperm(len(all_edges), generator=generator)[:edge_count]]
        all_triangles = torch.combinations(torch.arange(self.N), r=3)
        self.triangles = all_triangles[
            torch.randperm(len(all_triangles), generator=generator)[:triangle_count]
        ]
        return self

    # [中介变量] 为每条记忆累计所有已选单形上的局部相似度
    def _simplex_scores(self, cue: torch.Tensor, simplices: torch.Tensor) -> torch.Tensor:
        if len(simplices) == 0:
            return torch.zeros(self.P, dtype=torch.float64)
        # [张量] memory_parts:(P,S,order)，cue_parts:(S,order)
        memory_parts = self.patterns[:, simplices]
        cue_parts = cue.to(torch.float64)[simplices]
        dot = (memory_parts * cue_parts.unsqueeze(0)).sum(dim=2)
        # [实验控制] dot 先除单形维数，排除三元项仅因维度更大而得分更高
        if self.similarity == "dot":
            return (dot / simplices.shape[1]).sum(dim=1)
        # [数值稳定] cosine 分母下限防止零范数；二值输入通常不会触发
        if self.similarity == "cosine":
            denominator = (
                memory_parts.norm(dim=2) * cue_parts.norm(dim=1).unsqueeze(0)
            ).clamp_min(1e-12)
            return (dot / denominator).sum(dim=1)
        raise ValueError("similarity must be dot or cosine")

    def retrieve(self, cue: torch.Tensor) -> torch.Tensor:
        edge_scores = self._simplex_scores(cue, self.edges)
        triangle_scores = self._simplex_scores(cue, self.triangles)
        simplex_count = len(self.edges) + len(self.triangles)
        # [实验控制] 再除总单形数，四格共享同一总分尺度
        scores = (edge_scores + triangle_scores) / simplex_count
        return torch.softmax(self.beta * scores, dim=0) @ self.patterns


# [执行] 四格设计：pairwise/mixed × dot/cosine，其他变量全部共享
def run_factorial_h7(base_seed: int = 20260908) -> pd.DataFrame:
    rows = []
    # [实验控制] 四个组合一次性预注册，禁止只运行看起来最好的两格
    factors = [(0.0, "dot"), (0.0, "cosine"), (0.5, "dot"), (0.5, "cosine")]
    for set_index in range(4):
        data_seed = stable_seed(base_seed, set_index)
        memories = a1_make_independent_binary(64, 16, data_seed)
        models = {
            factor: ContinuousSimplicialAttention(
                factor[0], factor[1], structure_seed=stable_seed(base_seed, set_index, 99),
                beta=8.0,
            ).fit(memories.patterns)
            for factor in factors
        }
        for target_id in range(8):
            cue_seed = stable_seed(base_seed, set_index, target_id)
            # [输入] 同一 target 的四格条件读取同一个 25% 损坏 cue
            cue = c1_make_hamming_cue(memories.patterns[target_id], 0.25, cue_seed)
            for (fraction, similarity), model in models.items():
                final = model.retrieve(cue)
                rows.append({
                    "structure": "pairwise" if fraction == 0.0 else "mixed",
                    "similarity": similarity,
                    "pattern_set_id": set_index,
                    "target_id": target_id,
                    "top1_correct": e3_top1_memory(final, memories.patterns) == target_id,
                })
    return pd.DataFrame(rows)


h7_results = run_factorial_h7()
h7_results.to_json(
    artifact_root / "h7_factorial_raw.jsonl", orient="records", lines=True
)
h7_table = h7_results.groupby(["structure", "similarity"])["top1_correct"].mean().unstack()
p00 = h7_table.loc["pairwise", "dot"]
p01 = h7_table.loc["pairwise", "cosine"]
p10 = h7_table.loc["mixed", "dot"]
p11 = h7_table.loc["mixed", "cosine"]
# [观测·交互] 概率尺度 p11-p10-p01+p00；必须与四格原始率一起解释
interaction = p11 - p10 - p01 + p00
h7_table.plot(kind="bar", figsize=(7.4, 4.6), ylim=(0, 1), rot=0)
plt.ylabel("Top-1 identification rate")
plt.title("H7 complete 2×2 | structure × similarity")
plt.tight_layout()
plt.show()
display(Markdown(
    f"**本次运行的描述性结论**：概率尺度交互项为 `{interaction:+.3f}`。正值表示两项联合收益超过两个单独主效应的加和，负值表示抵消。\n\n"
    "**证据边界**：必须连同四格原始率一起解释；单个交互数不能证明通用协同，也不能替代论文的图像数据实验。"
))
display(h7_table.round(3))

## 实验 10：H4/H5 Curved/Explosive 的有效温度与迟滞

论文的核心状态反馈是 `beta_eff = beta / (1 + gamma*m²/2)`。这里复现对应的一维平均场动力学 `dm/dt = -m + tanh(beta_eff*m)`，并从低 beta 正向、从高 beta 反向延续稳定支，检验是否真的存在路径依赖。它是论文专属理论轨道，不与确定性 Top-1 主图混排。

实现公式对照本 notebook 内的函数；网络级随机 Glauber 实验可对照论文的 [official repository](https://github.com/MiguelAguilera/explosive-neural-networks)。

In [ ]:
# [机制] beta_eff=beta/(1+gamma*m²/2)，温度由当前 overlap 反馈调制
def effective_beta(beta: float, gamma: float, overlap: float) -> float:
    denominator = 1.0 + 0.5 * gamma * overlap**2
    # [失败边界] 分母非正时曲率参数进入非法域，返回 NaN 而不是继续画假曲线
    if denominator <= 0.0:
        return float("nan")
    return beta / denominator


# [动力学] 显式积分 dm/dt=-m+tanh(beta_eff*m)，并返回是否达到容差
def relax_curved_mean_field(
    initial_m: float, beta: float, gamma: float,
    *, dt: float = 0.05, tolerance: float = 1e-10, max_steps: int = 20_000,
) -> tuple[float, bool]:
    m = float(initial_m)
    for _ in range(max_steps):
        beta_eff = effective_beta(beta, gamma, m)
        if not np.isfinite(beta_eff):
            return float("nan"), False
        # [更新] 一步 Euler 积分；clip 只维护 overlap 的物理区间 [0,1]
        next_m = m + dt * (-m + np.tanh(beta_eff * m))
        next_m = float(np.clip(next_m, 0.0, 1.0))
        if abs(next_m - m) < tolerance:
            return next_m, True
        m = next_m
    return m, False


# [实验控制] 前一点的稳定解作为下一 beta 初值，保留扫描方向的路径依赖
def continuation_branch(beta_grid: np.ndarray, gamma: float, initial_m: float) -> pd.DataFrame:
    rows = []
    m = initial_m
    for beta in beta_grid:
        # [数值稳定] 给零支极小扰动，避免 gamma=0 因精确 m=0 产生伪迟滞
        if m < 1e-8:
            m = 1e-4
        m, converged = relax_curved_mean_field(m, float(beta), gamma)
        rows.append({"beta": beta, "gamma": gamma, "m": m, "converged": converged})
    return pd.DataFrame(rows)


# [机制对照] 记录参考反馈轨迹及它实际经历的 beta_eff 时间表
def feedback_trajectory(
    initial_m: float, beta: float, gamma: float,
    *, steps: int = 400, dt: float = 0.05,
) -> tuple[np.ndarray, np.ndarray]:
    overlaps = [float(initial_m)]
    beta_schedule = []
    for _ in range(steps):
        beta_eff = effective_beta(beta, gamma, overlaps[-1])
        beta_schedule.append(beta_eff)
        next_m = overlaps[-1] + dt * (
            -overlaps[-1] + np.tanh(beta_eff * overlaps[-1])
        )
        overlaps.append(float(np.clip(next_m, 0.0, 1.0)))
    return np.asarray(overlaps), np.asarray(beta_schedule)


# [反事实] 对扰动初态播放同一时间表，但不允许 beta 根据新状态反馈
def open_loop_trajectory(
    initial_m: float, beta_schedule: np.ndarray, *, dt: float = 0.05
) -> np.ndarray:
    overlaps = [float(initial_m)]
    for beta_eff in beta_schedule:
        next_m = overlaps[-1] + dt * (
            -overlaps[-1] + np.tanh(beta_eff * overlaps[-1])
        )
        overlaps.append(float(np.clip(next_m, 0.0, 1.0)))
    return np.asarray(overlaps)


beta_grid = np.linspace(0.2, 3.0, 120)
gamma_values = (0.0, -0.5, -1.0, -1.5)
branches = []
# [实验控制] gamma=0 是固定温度基线，负 gamma 是爆发式正反馈条件
for gamma in gamma_values:
    forward = continuation_branch(beta_grid, gamma, initial_m=1e-6).assign(direction="forward")
    backward = continuation_branch(beta_grid[::-1], gamma, initial_m=0.999).assign(direction="backward")
    branches.extend([forward, backward])
curved_branches = pd.concat(branches, ignore_index=True)

# [实验控制] reference 与 perturbed-feedback/matched-open-loop 构成 H4 三组对照
reference_feedback, matched_schedule = feedback_trajectory(
    initial_m=0.85, beta=0.8, gamma=-1.5
)
perturbed_feedback, _ = feedback_trajectory(
    initial_m=0.35, beta=0.8, gamma=-1.5
)
perturbed_open_loop = open_loop_trajectory(0.35, matched_schedule)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.4))
for (gamma, direction), group in curved_branches.groupby(["gamma", "direction"]):
    axes[0].plot(
        group["beta"], group["m"],
        linestyle="-" if direction == "forward" else "--",
        label=f"gamma={gamma:g}, {direction}",
    )
overlap_grid = np.linspace(0, 1, 200)
for gamma in gamma_values:
    axes[1].plot(
        overlap_grid,
        [effective_beta(1.0, gamma, m) for m in overlap_grid],
        label=f"gamma={gamma:g}",
    )
axes[2].plot(reference_feedback, label="feedback, reference initial")
axes[2].plot(perturbed_feedback, label="feedback, perturbed initial")
axes[2].plot(perturbed_open_loop, "--", label="matched open-loop schedule")
axes[0].set(xlabel="Control beta", ylabel="Stable overlap m", title="H5 forward/backward continuation")
axes[1].set(xlabel="Overlap m", ylabel="Effective beta (base beta=1)", title="H4 state-feedback temperature")
axes[2].set(xlabel="Integration step", ylabel="Overlap m", title="H4 feedback vs matched schedule")
axes[0].legend(fontsize=7, ncol=2)
axes[1].legend(fontsize=8)
axes[2].legend(fontsize=7)
fig.tight_layout()
plt.show()

wide = curved_branches.pivot_table(index=["gamma", "beta"], columns="direction", values="m").reset_index()
# [观测·数据] 同一 beta 的正反稳定支差值，是 H5 迟滞的直接量尺
wide["branch_gap"] = (wide["forward"] - wide["backward"]).abs()
gap_summary = wide.groupby("gamma", as_index=False)["branch_gap"].max()
curved_branches.to_json(
    artifact_root / "h45_curved_branches.jsonl", orient="records", lines=True
)
# [存储] 保存三条时间轨迹，图不能成为唯一证据
h4_counterfactual = pd.DataFrame({
    "step": np.arange(len(reference_feedback)),
    "reference_feedback": reference_feedback,
    "perturbed_feedback": perturbed_feedback,
    "perturbed_matched_open_loop": perturbed_open_loop,
})
h4_counterfactual.to_json(
    artifact_root / "h4_feedback_counterfactual.jsonl",
    orient="records", lines=True,
)
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(
        f"- gamma={row.gamma:g}：正反扫描最大分支差={row.branch_gap:.3f}。"
        for row in gap_summary.itertuples(index=False)
    )
    + "\n\n**证据边界**：平均场迟滞是机制证据，不等同于有限 N 随机网络中的统计显著性；未收敛点必须先排查，不能当作相变。"
))
display(gap_summary.round(3))

## 实验 11：H6 PSHN 分组数与 feature-to-prototype

下面给出论文尺度的 MNIST 入口：`M=10,000`，`k∈{7,16,28,112}`，右侧遮挡 `theta∈{0,.25,.35}`，10 次不同子采样。除了论文的“完美恢复数量”，还记录输出离目标实例和目标类别原型的误差，防止高阶模型输出原型却被单一指标掩盖。

默认 `RUN_PSHN_MNIST=False`，因为这是昂贵、会下载 MNIST 的论文专属实验；切换为 `True` 后才在 Colab 执行。关闭时不会生成伪结果。

In [ ]:
# [执行门] 默认关闭，确保普通 Run All 不会意外下载数据或启动论文尺度计算
RUN_PSHN_MNIST = False


# [实验控制] 论文尺度 M、k、遮挡率、10 次子采样和批大小集中登记
@dataclass(frozen=True)
class PSHNMNISTConfig:
    memory_count: int = 10_000
    query_count: int = 10_000
    groups: tuple[int, ...] = (7, 16, 28, 112)
    occlusion_levels: tuple[float, ...] = (0.0, 0.25, 0.35)
    trials: int = 10
    query_batch_size: int = 32
    base_seed: int = 20260909


# [更新] 论文 PSHN 一次读出的批量版；输出仍是 {-1,+1}
def pshn_batch_update(
    queries: torch.Tensor, memories: torch.Tensor, groups: int
) -> torch.Tensor:
    M, N = memories.shape
    batch = len(queries)
    group_size = N // groups
    X = memories.reshape(M, groups, group_size)
    Q = queries.reshape(batch, groups, group_size)
    # [中介变量] (B,k,N/k) 与 (M,k,N/k)→(B,M,k) 分组相关度
    correlations = torch.einsum("bkg,Mkg->bMk", Q, X)
    # [数值稳定] k=112 时直接连乘会溢出，改在符号/对数域构造等价相对系数
    nonzero = correlations != 0
    safe_sign = torch.where(nonzero, torch.sign(correlations), torch.ones_like(correlations))
    safe_log_abs = torch.where(
        nonzero, torch.log(correlations.abs()), torch.zeros_like(correlations)
    )
    # [中介变量] 排除第 g 组后仍含零相关度，则该 memory-group 系数必须为零
    zero_count = (~nonzero).sum(dim=2, keepdim=True)
    valid = zero_count - (~nonzero).to(torch.int64) == 0
    sign_without = safe_sign.prod(dim=2, keepdim=True) * safe_sign
    log_without = safe_log_abs.sum(dim=2, keepdim=True) - safe_log_abs
    masked_log = torch.where(valid, log_without, torch.full_like(log_without, -torch.inf))
    # [数值稳定] 每条 query 减去共同最大 log 系数；只乘正比例常数，不改变局部场符号
    scale = masked_log.amax(dim=(1, 2), keepdim=True)
    scale = torch.where(torch.isfinite(scale), scale, torch.zeros_like(scale))
    C = torch.where(valid, sign_without * torch.exp(log_without - scale), 0.0)
    # [更新] C:(B,M,k) 与 X:(M,k,N/k)→fields:(B,k,N/k)
    fields = torch.einsum("bMk,Mkg->bkg", C, X)
    output = torch.sign(fields).reshape(batch, N)
    ties = output == 0
    output[ties] = queries[ties]
    return output


# [输入] 按论文把图像右侧连续 100*theta% 像素设为 -1
def right_occlusion(patterns: torch.Tensor, theta: float) -> torch.Tensor:
    images = patterns.reshape(-1, 28, 28).clone()
    width = int(round(28 * theta))
    if width:
        images[:, :, 28 - width:] = -1
    return images.reshape(-1, 784)


# [执行] 每个 trial 从完整 70K MNIST 中重新无放回抽取 10K 记忆
def run_pshn_mnist(config: PSHNMNISTConfig) -> pd.DataFrame:
    from torchvision.datasets import MNIST

    # [环境] 论文尺度优先使用 Colab GPU；CPU 仍可运行但会很慢
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train = MNIST("/content/data", train=True, download=True)
    test = MNIST("/content/data", train=False, download=True)
    pixels = torch.cat([train.data, test.data]).reshape(-1, 784)
    labels = torch.cat([train.targets, test.targets])
    # [输入] 像素阈值化为 {-1,+1}，与论文实验编码一致
    binary = torch.where(pixels >= 128, 1.0, -1.0).to(torch.float32)
    rows = []
    for trial in range(config.trials):
        generator = make_generator(stable_seed(config.base_seed, trial))
        # [实验控制] trial 改变 10K 子样本；同一 trial 的全部 k/theta 共享样本
        indices = torch.randperm(len(binary), generator=generator)[:config.memory_count]
        memories = binary[indices].to(device)
        memory_labels = labels[indices].to(device)
        query_count = min(config.query_count, config.memory_count)
        targets = memories[:query_count]
        target_labels = memory_labels[:query_count]
        # [中介变量] 每个数字类别的符号均值原型，用于 feature-to-prototype 诊断
        prototypes = torch.stack([
            torch.sign(memories[memory_labels == digit].mean(dim=0))
            for digit in range(10)
        ])
        for groups in config.groups:
            if 784 % groups:
                raise ValueError(f"groups={groups} does not divide 784")
            for theta in config.occlusion_levels:
                # [输入] 遮挡在 group 循环内固定生成；同一 k 的 query 顺序不变
                queries = right_occlusion(targets, theta)
                for start in range(0, query_count, config.query_batch_size):
                    stop = min(start + config.query_batch_size, query_count)
                    output = pshn_batch_update(queries[start:stop], memories, groups)
                    batch_targets = targets[start:stop]
                    batch_labels = target_labels[start:stop]
                    # [观测·数据] exact 要求 784 位全对，严格复核论文完美恢复量
                    exact = torch.all(output == batch_targets, dim=1)
                    instance_error = (output != batch_targets).float().mean(dim=1)
                    # [观测·机制] 与类别原型误差和实例误差并列，防止原型化输出被算成容量收益
                    prototype_error = (
                        output != prototypes[batch_labels]
                    ).float().mean(dim=1)
                    # [观测·错误] 原型最近类别用于识别跨类别误检，不能只看平均像素误差
                    prototype_prediction = torch.argmax(output @ prototypes.T, dim=1)
                    for offset in range(stop - start):
                        rows.append({
                            "trial": trial,
                            "groups": groups,
                            "occlusion": theta,
                            "query_id": start + offset,
                            "exact_recall": bool(exact[offset].item()),
                            "instance_error": float(instance_error[offset].item()),
                            "prototype_error": float(prototype_error[offset].item()),
                            "prototype_class_correct": bool(
                                prototype_prediction[offset] == batch_labels[offset]
                            ),
                        })
    return pd.DataFrame(rows)


# [执行门] 只有用户在 Colab 显式打开开关，下面才下载并产生 H6 结果
if RUN_PSHN_MNIST:
    pshn_mnist = run_pshn_mnist(PSHNMNISTConfig())
    pshn_mnist.to_json(
        artifact_root / "h6_pshn_mnist_raw.jsonl", orient="records", lines=True
    )
    pshn_summary = pshn_mnist.groupby(["groups", "occlusion"], as_index=False).agg(
        exact_recall=("exact_recall", "mean"),
        instance_error=("instance_error", "mean"),
        prototype_error=("prototype_error", "mean"),
        prototype_class_correct=("prototype_class_correct", "mean"),
    )
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4))
    for theta, group in pshn_summary.groupby("occlusion"):
        axes[0].plot(group["groups"], group["exact_recall"], marker="o", label=f"theta={theta:g}")
        axes[1].plot(
            group["groups"], group["instance_error"] - group["prototype_error"],
            marker="o", label=f"theta={theta:g}",
        )
    axes[0].set(xlabel="PSHN groups k", ylabel="Perfect recovery rate", title="H6 MNIST instance recovery")
    axes[1].set(
        xlabel="PSHN groups k", ylabel="Instance error - prototype error",
        title="H6 feature-to-prototype diagnostic",
    )
    for axis in axes:
        axis.set_xscale("log", base=2)
        axis.legend()
    fig.tight_layout()
    plt.show()
    display(Markdown(
        "**本次运行的描述性结论**：左图复核完美恢复，右图为正时表示输出比目标实例更靠近类别原型。\n\n"
        "**证据边界**：必须同时检查类别正确率与同类误检；不同 k 的改善不能单独归因为容量。"
    ))
    display(pshn_summary.round(4))
else:
    display(Markdown(
        "**H6 尚未执行**：将 `RUN_PSHN_MNIST` 改为 `True` 后，Colab 才会下载 MNIST 并运行论文尺度实验；当前没有图，也没有数值结论。"
    ))

## 12. 读图顺序与结论门

1. 先看 U2/U3/U4 的共同 Top-1 曲线，确认相同数据、线索和 trial 配对。
2. 再看 U1 固定点与 U5/U6 动力学；不适用的模型不进入对应排名。
3. 用 U7 检查 native 资源差异，再用 H1–H3 的同参数消融判断高阶结构收益。
4. H4–H7 只解释机制，不把论文专属数字混入统一冠军榜。

只有当原始记录、失败行、随机种子、预算与图下结论一一对应时，才允许写“本次运行支持”。有限扫描不支持线性、多项式、指数或双指数容量的渐近声明。